# Whisker-Pole Contact Classifier — Inference

Run the trained EfficientNet-B3 on a video and produce:
1. **Per-frame CSV** — every frame with its contact prediction (0/1) and probability
2. **Interval CSV** — contiguous contact regions in `Start,End` format

In [1]:
import sys, os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

INFERENCE_DIR = os.path.dirname(os.path.abspath("__file__"))
if INFERENCE_DIR not in sys.path:
    sys.path.insert(0, INFERENCE_DIR)

from utils import (
    load_model,
    get_inference_transforms,
    VideoFrameDataset,
    run_inference,
    frames_to_intervals,
)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch 2.5.0+cu124
CUDA available: True
GPU: NVIDIA L40S


## 1 — Configuration

In [2]:
# ========================  PATHS  ========================

VIDEO_PATH = "/home/mvdokh/test/tracking - Copy/whiskers/whisker_tracking/sequential_whisker/sequential_CNXT_n1/102225_1/102225_1.mp4"


CHECKPOINT_PATH = os.path.join(
    os.path.dirname(INFERENCE_DIR),
    "Training", "checkpoints", "contact_classifier.pt"
)

# Output CSVs will be saved here
OUTPUT_DIR = os.path.join(INFERENCE_DIR, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================  SETTINGS  ========================

IMG_SIZE      = 256
BATCH_SIZE    = 64       # can be larger than training since no gradients
NUM_WORKERS   = 2        # try higher to parallelize video decoding
THRESHOLD     = 0.5      # probability threshold for contact
START_FRAME   = 0
END_FRAME     = 250_000  # only first 250k frames (second half has no pole)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

Device: cuda
Checkpoint: /orcd/home/002/mvdokh/Deep-Learning/Contact Classification/Training/checkpoints/contact_classifier.pt
Output dir: /orcd/home/002/mvdokh/Deep-Learning/Contact Classification/Inference/results


## 2 — Load Model

In [3]:
model = load_model(CHECKPOINT_PATH, DEVICE)

Loaded checkpoint from: /orcd/home/002/mvdokh/Deep-Learning/Contact Classification/Training/checkpoints/contact_classifier.pt
  Config: {'img_size': 256, 'batch_size': 32, 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'freeze_backbone': False, 'model_arch': 'efficientnet_b3'}
  Epoch: 22
  Val loss: 0.0007


## 3 — Create Video Dataset & DataLoader

In [4]:
transform = get_inference_transforms(IMG_SIZE)

dataset = VideoFrameDataset(
    video_path=VIDEO_PATH,
    start_frame=START_FRAME,
    end_frame=END_FRAME,
    transform=transform,
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Total frames to process: {len(dataset):,}")
print(f"Batches: {len(dataloader):,}")

import time


def profile_video_inference(model, dataloader, device, n_batches: int = 20):
    """Profile a few batches of the video dataloader.

    Prints average time per batch spent in:
      - data loading + video decode + transforms
      - model forward pass
    and an approximate end-to-end FPS.
    """
    model.eval()
    data_iter = iter(dataloader)

    total_loader = 0.0
    total_model = 0.0
    total_batches = 0
    total_frames = 0

    for _ in range(n_batches):
        t0 = time.time()
        try:
            images, frame_nums = next(data_iter)
        except StopIteration:
            break
        t1 = time.time()

        bsz = images.size(0)
        total_frames += bsz

        images = images.to(device, non_blocking=True)

        if device.type == "cuda":
            torch.cuda.synchronize()
        t2 = time.time()

        with torch.no_grad():
            _ = model(images).squeeze(1)

        if device.type == "cuda":
            torch.cuda.synchronize()
        t3 = time.time()

        total_loader += (t1 - t0)
        total_model += (t3 - t2)
        total_batches += 1

    if total_batches == 0:
        print("No batches available to profile.")
        return

    avg_loader = total_loader / total_batches
    avg_model = total_model / total_batches
    total_time = total_loader + total_model

    print(f"Profiled {total_batches} batches ({total_frames} frames)")
    print(f"Avg DataLoader+decode+transforms time / batch: {avg_loader:.3f} s")
    print(f"Avg model forward time / batch:                {avg_model:.3f} s")
    share_data = 100 * avg_loader / (avg_loader + avg_model)
    share_model = 100 * avg_model / (avg_loader + avg_model)
    print(f"Share of time in data vs model:                {share_data:.1f}% data, {share_model:.1f}% model")
    if total_time > 0:
        fps = total_frames / total_time
        print(f"Approx end-to-end throughput:                 {fps:.1f} frames/s")


# Example usage (after creating `model` and `dataloader`):
profile_video_inference(model, dataloader, DEVICE, n_batches=20)

VideoFrameDataset: frames 0–249999 (250,000 frames)
Total frames to process: 250,000
Batches: 3,907
Profiled 20 batches (1280 frames)
Avg DataLoader+decode+transforms time / batch: 0.187 s
Avg model forward time / batch:                0.128 s
Share of time in data vs model:                59.3% data, 40.7% model
Approx end-to-end throughput:                 203.4 frames/s


## 4 — Run Inference

In [5]:
results_df = run_inference(model, dataloader, DEVICE, threshold=THRESHOLD)

n_contact = (results_df["contact"] == 1).sum()
n_no_contact = (results_df["contact"] == 0).sum()
print(f"\nTotal frames   : {len(results_df):,}")
print(f"Contact frames : {n_contact:,} ({100*n_contact/len(results_df):.1f}%)")
print(f"No contact     : {n_no_contact:,} ({100*n_no_contact/len(results_df):.1f}%)")
print()
results_df.head(10)

Inference:   0%|          | 0/3907 [00:00<?, ?it/s]

Inference:   2%|▏         | 65/3907 [00:13<13:25,  4.77it/s] 


KeyboardInterrupt: 

## 5 — Convert to Intervals & Save CSVs

In [6]:
# Per-frame CSV
per_frame_path = os.path.join(OUTPUT_DIR, "per_frame_predictions.csv")
results_df.to_csv(per_frame_path, index=False)
print(f"Saved per-frame predictions → {per_frame_path}")

# Contact intervals CSV
intervals_df = frames_to_intervals(results_df, label_col="contact", label_val=1)
intervals_path = os.path.join(OUTPUT_DIR, "contact_intervals.csv")
intervals_df.to_csv(intervals_path, index=False)
print(f"Saved contact intervals     → {intervals_path}")
print(f"\nFound {len(intervals_df)} contact intervals")
print()
intervals_df.head(20)

Saved per-frame predictions → /orcd/home/002/mvdokh/Deep-Learning/Contact Classification/Inference/results/per_frame_predictions.csv
Saved contact intervals     → /orcd/home/002/mvdokh/Deep-Learning/Contact Classification/Inference/results/contact_intervals.csv

Found 3463 contact intervals



,Start,End
0,9649,9653
1,9660,9661
2,9664,9668
3,9675,9675
4,9684,9691
5,9727,9728
6,9741,9743
7,9755,9757
8,9759,9782
9,9788,9790


## 6 — Quick Summary

In [7]:
# Interval length stats
intervals_df["length"] = intervals_df["End"] - intervals_df["Start"] + 1

print(f"Contact intervals: {len(intervals_df)}")
print(f"Total contact frames: {intervals_df['length'].sum():,}")
print(f"Avg interval length: {intervals_df['length'].mean():.1f} frames")
print(f"Min interval length: {intervals_df['length'].min()} frames")
print(f"Max interval length: {intervals_df['length'].max()} frames")
print(f"Median interval length: {intervals_df['length'].median():.0f} frames")

Contact intervals: 3463
Total contact frames: 101,543
Avg interval length: 29.3 frames
Min interval length: 1 frames
Max interval length: 3227 frames
Median interval length: 8 frames
